# UNI2-h Training on Google Colab (Resume-Safe)

This notebook runs `scripts/train_uni2h_preprocessed_colab.py` on the **same preprocessed H5** layout as Virchow2 (`train_x/y.h5`, `valid_x/y.h5`) and stores checkpoints on Google Drive so training can resume after disconnects.

**Before first run:** accept the gated license at [MahmoodLab/UNI2-h](https://huggingface.co/MahmoodLab/UNI2-h) and set `HF_TOKEN` (or `huggingface-cli login`).

**Not a drop-in model swap:** UNI2-h uses a **1536-d** CLS pooled embedding (ViT-H/14 + 8 reg tokens) and SwiGLU `timm` load kwargs (see script `get_embedding` / `Uni2hClassifier`). Default batch size is **48** (ViT-H is heavier than Virchow2 on Colab).

In [ ]:
!fusermount -u /content/drive || true
!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!git clone https://github.com/LeenRayess/GP_ECG.git GP_ECG
%cd /content/GP_ECG

If you already uploaded your repo into Drive instead of cloning, skip previous cell and set `REPO_DIR` below to that path.

In [ ]:
REPO_DIR = '/content/GP_ECG'
PREPROCESSED_DIR = '/content/drive/MyDrive/GP_ECG_DATA/preprocessed_macenko_benchmark_style'
RUN_DIR = '/content/drive/MyDrive/GP_ECG_RUNS/uni2h_macenko_bench_run_01'
HF_TOKEN = ''  # paste from https://huggingface.co/settings/tokens if needed

import os
os.makedirs(RUN_DIR, exist_ok=True)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN

In [ ]:
import os
import shutil

# Copy preprocessed .h5 from Google Drive → Colab local SSD (faster reads). RUN_DIR stays on Drive.
LOCAL_PREPROCESSED_DIR = "/content/local_preprocessed_h5"

_src = PREPROCESSED_DIR
if _src.startswith("/content/drive/"):
    if not os.path.isdir(_src):
        raise FileNotFoundError("Drive path not found — mount Drive and check PREPROCESSED_DIR:\n  " + _src)
    if not os.path.exists(LOCAL_PREPROCESSED_DIR):
        print("Copying preprocessed data Drive → local SSD (may take several minutes)...")
        print("  from:", _src)
        shutil.copytree(_src, LOCAL_PREPROCESSED_DIR)
        print("  done:", LOCAL_PREPROCESSED_DIR)
    else:
        print("Using existing local copy:", LOCAL_PREPROCESSED_DIR)
    PREPROCESSED_DIR = LOCAL_PREPROCESSED_DIR
else:
    print("PREPROCESSED_DIR is not under /content/drive/ — using as-is:", _src)

In [ ]:
# import os
# print(os.path.isdir("/content/GP_ECG"))

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import torch
print(torch.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
%cd {REPO_DIR}
!python -m pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install timm h5py tqdm huggingface_hub scikit-learn

UNI2-h is gated: accept the license at https://huggingface.co/MahmoodLab/UNI2-h then login with your HF token.

In [ ]:
# !huggingface-cli login

In [ ]:
%cd {REPO_DIR}
!python scripts/train_uni2h_preprocessed_colab.py \
  --preprocessed-dir "{PREPROCESSED_DIR}" \
  --out-dir "{RUN_DIR}" \
  --epochs 10 \
  --batch-size 48 \
  --num-workers 0 \
  --resume \
  --save-every-epoch-copy

In [ ]:
import json, os
print('Run dir:', RUN_DIR)
artifacts = [
    'checkpoint_last.pt', 'model_best.pt', 'metrics_history.json', 'metrics_final.json',
    'metrics_final_detailed.json', 'temperature_fit.json', 'run_config.json', 'run_manifest.json',
    'run_progress.json', 'val_predictions.npz',
]
for name in artifacts:
    p = os.path.join(RUN_DIR, name)
    print(('OK ' if os.path.exists(p) else 'MISSING '), p)

hist = os.path.join(RUN_DIR, 'metrics_history.json')
if os.path.exists(hist):
    with open(hist, 'r', encoding='utf-8') as f:
        rows = json.load(f)
    if rows:
        print('Last epoch record:', rows[-1])

det = os.path.join(RUN_DIR, 'metrics_final_detailed.json')
if os.path.exists(det):
    with open(det, 'r', encoding='utf-8') as f:
        d = json.load(f)
    print('metrics_final_detailed keys:', list(d.keys()))